### Question 1

In [9]:
import pandas as pd
import numpy as np

url = 'https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv'
df = pd.read_csv(url)

for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = df[col].fillna('NA')
    else:
        df[col] = df[col].fillna(0.0)

print(df['industry'].mode().iloc[0])


technology


### Question 2

In [10]:
import pandas as pd
import numpy as np

url = 'https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv'
df = pd.read_csv(url)

for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = df[col].fillna('NA')
    else:
        df[col] = df[col].fillna(0.0)

num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
corr = df[num_cols].corr()
choices = [
    ('interaction_count', 'lead_score'),
    ('number_of_courses_viewed', 'lead_score'),
    ('number_of_courses_viewed', 'interaction_count'),
    ('annual_income', 'interaction_count'),
]
result = max(choices, key=lambda p: abs(corr.loc[p[0], p[1]]))
print(result)
print(corr.loc[result[0], result[1]])


('interaction_count', 'lead_score')
0.9157457980418697


### Question 3

In [11]:
import pandas as pd
from sklearn.feature_selection import mutual_info_classif
from sklearn.model_selection import train_test_split

url = 'https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv'
df = pd.read_csv(url)

for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = df[col].fillna('NA')
    else:
        df[col] = df[col].fillna(0.0)

df_full_train, _ = train_test_split(df, test_size=0.2, random_state=42)
df_train, _ = train_test_split(df_full_train, test_size=0.25, random_state=42)

X_train = df_train.drop(columns=['converted'])
y_train = df_train['converted']
cat_cols = X_train.select_dtypes(include=['object']).columns.tolist()
mi_scores = {}

for col in cat_cols:
    vals = X_train[col].astype('category').cat.codes.to_numpy().reshape(-1, 1)
    score = mutual_info_classif(vals, y_train, discrete_features=[0], random_state=42)[0]
    mi_scores[col] = score

print({k: round(v, 2) for k, v in mi_scores.items()})
print(max(mi_scores, key=mi_scores.get))


{'lead_source': np.float64(0.03), 'industry': np.float64(0.0), 'employment_status': np.float64(0.02), 'location': np.float64(0.0)}
lead_source


### Question 4

In [12]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

url = 'https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv'
df = pd.read_csv(url)

for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = df[col].fillna('NA')
    else:
        df[col] = df[col].fillna(0.0)

df_full_train, _ = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=42)
X_train = df_train.drop(columns=['converted'])
y_train = df_train['converted']
X_val = df_val.drop(columns=['converted'])
y_val = df_val['converted']

cat_cols = X_train.select_dtypes(include=['object']).columns.tolist()
num_cols = X_train.select_dtypes(exclude=['object']).columns.tolist()

preprocessor = ColumnTransformer([
    ('num', 'passthrough', num_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols),
])

model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('model', model),
])

pipe.fit(X_train, y_train)
acc = accuracy_score(y_val, pipe.predict(X_val))
print(round(acc, 2))


0.65


### Question 5

In [13]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

url = 'https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv'
df = pd.read_csv(url)

for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = df[col].fillna('NA')
    else:
        df[col] = df[col].fillna(0.0)

df_full_train, _ = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=42)

X_train = df_train.drop(columns=['converted'])
y_train = df_train['converted']
X_val = df_val.drop(columns=['converted'])
y_val = df_val['converted']
cat_cols = X_train.select_dtypes(include=['object']).columns.tolist()

base_preprocessor = ColumnTransformer([
    ('num', 'passthrough', X_train.select_dtypes(exclude=['object']).columns.tolist()),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols),
])

base_model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
base_pipe = Pipeline([
    ('preprocessor', base_preprocessor),
    ('model', base_model),
])
base_pipe.fit(X_train, y_train)
base_acc = accuracy_score(y_val, base_pipe.predict(X_val))

feature_diffs = {}
for feature in ['lead_source', 'number_of_courses_viewed', 'interaction_count']:
    feats = [col for col in X_train.columns if col != feature]
    X_train_ex = X_train[feats]
    X_val_ex = X_val[feats]
    cat_ex = [c for c in feats if c in cat_cols]
    num_ex = [c for c in feats if c not in cat_cols]

    pre_ex = ColumnTransformer([
        ('num', 'passthrough', num_ex),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_ex),
    ])
    model_ex = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
    pipe_ex = Pipeline([
        ('preprocessor', pre_ex),
        ('model', model_ex),
    ])
    pipe_ex.fit(X_train_ex, y_train)
    acc_ex = accuracy_score(y_val, pipe_ex.predict(X_val_ex))
    feature_diffs[feature] = base_acc - acc_ex
    print(feature, 'acc_without_feature=', acc_ex, 'diff=', feature_diffs[feature])

print('best_feature=', min(feature_diffs, key=lambda k: abs(feature_diffs[k])))


lead_source acc_without_feature= 0.642 diff= 0.0030000000000000027
number_of_courses_viewed acc_without_feature= 0.643 diff= 0.0020000000000000018
interaction_count acc_without_feature= 0.601 diff= 0.04400000000000004
best_feature= number_of_courses_viewed


### Question 6

In [14]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

url = 'https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv'
df = pd.read_csv(url)

for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = df[col].fillna('NA')
    else:
        df[col] = df[col].fillna(0.0)

df_full_train, _ = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=42)

X_train = df_train.drop(columns=['converted'])
y_train = df_train['converted']
X_val = df_val.drop(columns=['converted'])
y_val = df_val['converted']

cat_cols = X_train.select_dtypes(include=['object']).columns.tolist()
num_cols = X_train.select_dtypes(exclude=['object']).columns.tolist()

results = {}
for C in [0.000001, 0.00001, 0.0001, 0.001]:
    preprocessor = ColumnTransformer([
        ('num', 'passthrough', num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols),
    ])
    model = LogisticRegression(solver='liblinear', C=C, max_iter=1000, random_state=42)
    pipe = Pipeline([
        ('preprocessor', preprocessor),
        ('model', model),
    ])
    pipe.fit(X_train, y_train)
    acc = accuracy_score(y_val, pipe.predict(X_val))
    results[C] = acc
    print(C, round(acc, 3))

print('best_C=', min(results, key=lambda c: (-results[c], c)))


1e-06 0.598
1e-05 0.598
0.0001 0.613
0.001 0.645
best_C= 0.001
